# Lesson 40: Sentiment Analysis

**Data Science with Python**

This notebook covers sentiment analysis techniques for financial text data.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/data-science-notebooks/blob/main/L40_Sentiment_Analysis/colab/L40_Sentiment_Analysis_Notebook.ipynb)

## 1. Setup and Imports

In [ ]:
# Install required packages (run once)
# !pip install vaderSentiment textblob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print("Libraries loaded successfully!")

## 2. Sample Financial Headlines

In [ ]:
# Financial headlines with sentiment labels
headlines = [
    ("Stock market soars to record highs", 1),
    ("Company reports strong quarterly earnings", 1),
    ("Investors optimistic about tech sector growth", 1),
    ("Market crashes amid economic uncertainty", 0),
    ("Layoffs announced as profits decline", 0),
    ("Trade war fears spark market sell-off", 0),
    ("Revenue exceeds analyst expectations", 1),
    ("Stock price plummets after earnings miss", 0),
    ("Merger deal boosts shareholder value", 1),
    ("Recession fears grow as indicators weaken", 0)
]

texts = [h[0] for h in headlines]
labels = [h[1] for h in headlines]

df = pd.DataFrame({'headline': texts, 'sentiment': labels})
df['sentiment_label'] = df['sentiment'].map({1: 'Positive', 0: 'Negative'})
df

## 3. Lexicon-Based: VADER

In [ ]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    
    # Initialize VADER
    analyzer = SentimentIntensityAnalyzer()
    
    # Analyze each headline
    print("VADER Sentiment Scores:")
    print("-" * 60)
    for text in texts[:5]:
        scores = analyzer.polarity_scores(text)
        print(f"Text: {text[:40]}...")
        print(f"  Compound: {scores['compound']:.3f}")
        print()
except ImportError:
    print("Install vaderSentiment: pip install vaderSentiment")

## 4. Lexicon-Based: TextBlob

In [ ]:
try:
    from textblob import TextBlob
    
    print("TextBlob Sentiment Scores:")
    print("-" * 60)
    for text in texts[:5]:
        blob = TextBlob(text)
        print(f"Text: {text[:40]}...")
        print(f"  Polarity: {blob.sentiment.polarity:.3f}")
        print(f"  Subjectivity: {blob.sentiment.subjectivity:.3f}")
        print()
except ImportError:
    print("Install textblob: pip install textblob")

## 5. ML-Based: TF-IDF + Logistic Regression

In [ ]:
# Expand dataset for training
training_data = [
    ("Stock prices surge on positive news", 1),
    ("Company announces record profits", 1),
    ("Investors bullish on market outlook", 1),
    ("Strong growth in tech sector", 1),
    ("Dividend increase pleases shareholders", 1),
    ("Market tumbles on bad economic data", 0),
    ("Stock plunges after disappointing earnings", 0),
    ("Recession fears grip investors", 0),
    ("Company faces bankruptcy risk", 0),
    ("Layoffs signal trouble ahead", 0)
] + headlines

X = [d[0] for d in training_data]
y = [d[1] for d in training_data]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# TF-IDF + Logistic Regression
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

clf = LogisticRegression()
clf.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = clf.predict(X_test_tfidf)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

## 6. Predict New Headlines

In [ ]:
# Test on new headlines
new_headlines = [
    "Earnings beat expectations by wide margin",
    "Market crashes on inflation fears",
    "CEO announces major acquisition"
]

new_tfidf = tfidf.transform(new_headlines)
predictions = clf.predict(new_tfidf)
probabilities = clf.predict_proba(new_tfidf)

print("Predictions on New Headlines:")
for headline, pred, prob in zip(new_headlines, predictions, probabilities):
    sentiment = 'Positive' if pred == 1 else 'Negative'
    confidence = max(prob) * 100
    print(f"  {headline}")
    print(f"    -> {sentiment} ({confidence:.1f}% confidence)")
    print()

## 7. Exercise: Aggregate Sentiment Signal

Calculate a daily sentiment score from multiple headlines and visualize it.

<details>
<summary>Click for Solution</summary>

```python
# Simulated daily headlines
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=30, freq='D')

# Generate random sentiment scores
daily_sentiment = np.random.randn(30).cumsum() * 0.1 + 0.5
daily_sentiment = np.clip(daily_sentiment, -1, 1)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dates, daily_sentiment, 'b-', linewidth=2)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(dates, daily_sentiment, 0, 
                where=daily_sentiment > 0, alpha=0.3, color='green', label='Positive')
ax.fill_between(dates, daily_sentiment, 0, 
                where=daily_sentiment < 0, alpha=0.3, color='red', label='Negative')
ax.set_xlabel('Date')
ax.set_ylabel('Aggregate Sentiment')
ax.set_title('Daily News Sentiment Signal')
ax.legend()
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Your solution here


## 8. Exercise: Compare VADER vs ML

Compare VADER predictions to ML predictions on test headlines.

<details>
<summary>Click for Solution</summary>

```python
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Compare on test set
results = []
for text, true_label in zip(X_test, y_test):
    # VADER
    vader_score = analyzer.polarity_scores(text)['compound']
    vader_pred = 1 if vader_score > 0 else 0
    
    # ML
    ml_pred = clf.predict(tfidf.transform([text]))[0]
    
    results.append({
        'text': text[:30] + '...',
        'true': true_label,
        'vader': vader_pred,
        'ml': ml_pred
    })

results_df = pd.DataFrame(results)
print(results_df)

# Accuracy comparison
vader_acc = (results_df['true'] == results_df['vader']).mean()
ml_acc = (results_df['true'] == results_df['ml']).mean()
print(f"\nVADER Accuracy: {vader_acc:.2%}")
print(f"ML Accuracy: {ml_acc:.2%}")
```
</details>

In [ ]:
# Your solution here


## Key Takeaways

1. **Lexicon-based** (VADER, TextBlob): No training needed, works out-of-box
2. **ML-based** (TF-IDF + classifier): Learns from data, often more accurate
3. **VADER** optimized for social media, uses intensity modifiers
4. **Aggregate sentiment** from multiple sources = trading signal
5. **Domain-specific** models (FinBERT) outperform general ones for finance